# LCEL 链式调用

## 顺序链

In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.chat_models import ChatTongyi
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, PromptTemplate

load_dotenv(override=True)
qwen_api_key = os.getenv("DASHSCOPE_API_KEY")

template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("你是{name}，请回答问题"),
    ("human", "你能干什么"),
    ("ai", "我可以做很多事情，比如回答问题，写代码，写文章，写书"),
    ("human", "{question}是什么")
])

model = ChatTongyi(
    model="qwen-plus",
    max_retries=2,
    api_key=qwen_api_key,
)
parser = StrOutputParser()
chain = template | model | parser
res = chain.invoke({"name": "小黑", "question": "langchain"})
print(res)


## 分支链 if/elif/else

In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.chat_models import ChatTongyi
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableParallel

load_dotenv(override=True)
qwen_api_key = os.getenv("DASHSCOPE_API_KEY")

model = ChatTongyi(
    model="qwen-plus",
    max_retries=2,
    api_key=qwen_api_key,
)
parser = StrOutputParser()
tech_chain = ChatPromptTemplate.from_template("请用技术语言回答：{question}") | model | parser
normal_chain = ChatPromptTemplate.from_template("请用通俗方式回答：{question}") | model | parser

branch_chain = RunnableBranch(
    (lambda x: "code" in x["question"], tech_chain),
    normal_chain  # 默认分支
)

res = branch_chain.invoke({"question": "langchain code"})
print(res)


## 并行链 RunnableParallel

In [ ]:
import os
from dotenv import load_dotenv
from langchain_core.runnables import RunnableParallel
from langchain_core.prompts import PromptTemplate
from langchain_community.chat_models import ChatTongyi
from langchain_core.output_parsers import StrOutputParser

load_dotenv(override=True)
qwen_api_key = os.getenv("DASHSCOPE_API_KEY")

model = ChatTongyi(
    model="qwen-plus",
    max_retries=2,
    api_key=qwen_api_key,
)
parser = StrOutputParser()

summary_chain = PromptTemplate.from_template("请总结这段文字：{text}") | model | parser
keyword_chain = PromptTemplate.from_template("提取3个关键词：{text}") | model | parser
classify_chain = PromptTemplate.from_template("判断文本类别：{text}") | model | parser

# 创建并行链
parallel_chain = RunnableParallel({
    "summary": summary_chain,
    "keywords": keyword_chain,
    "classify": classify_chain
})

text = """
LangChain是大模型应用开发框架，支持链式调用、RAG检索、智能体等功能，
广泛用于文档问答、内容创作、自动化流程等场景。
"""

result = parallel_chain.invoke({"text": text})
print(result)

# 链式调用高级用法

## RunnableLambda  
让普通 Python 函数能放进 | 管道链里执行。

In [ ]:
import os
from dotenv import load_dotenv
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import PromptTemplate
from langchain_community.chat_models import ChatTongyi
from langchain_core.output_parsers import StrOutputParser

load_dotenv(override=True)
qwen_api_key = os.getenv("DASHSCOPE_API_KEY")

model = ChatTongyi(
    model="qwen-plus",
    max_retries=2,
    api_key=qwen_api_key,
)
parser = StrOutputParser()


def upper(_text):
    return _text.upper()


chain = (
        PromptTemplate.from_template("你是一个翻译官,翻译:{text}")
        | model
        | parser
        | RunnableLambda(upper)  #包装成链节点
)
res = chain.invoke({"text": "骑虎难下！"})
print(res)

## RunnablePassthrough 
原样传递，不修改数据


In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template("你是{role}")
chain = prompt | {"res": RunnablePassthrough()}  #把纯文本包装成指定字段
res = chain.invoke({'role': "engine"})
print(res)

{'res': ChatPromptValue(messages=[HumanMessage(content='你是engine', additional_kwargs={}, response_metadata={})])}
